<a href="https://colab.research.google.com/github/RasanaPJ/Deep-Learning/blob/master/MCP_Integration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MCP Integration with LLM

This notebook walks through **three ways an LLM can reach beyond its training data**, in order of increasing control:

1. **No tools** — the model answers from its weights alone. We'll see it correctly refuse to answer a 2026 question because its knowledge is stale.
2. **Remote MCP server (Tavily)** — we hand the model a hosted MCP server URL and let it call a search tool over HTTP. We don't run any server ourselves; Tavily hosts it.

In [1]:
!pip install -q markdownify langchain langchain-groq --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.9 MB/s eta 0:00:00


In [2]:
import openai
import os
from getpass import getpass

## API Keys via `getpass`

`getpass` reads input without echoing it to the screen, so keys won't leak into cell outputs or notebook checkpoints. We'll prompt for one key now (Groq) and another later (Tavily) right before it's needed.

In [3]:
api_key = getpass("GROQ_API_KEY")

GROQ_API_KEY··········


**Groq through the OpenAI SDK.** Groq exposes an OpenAI-compatible REST API. By setting `base_url="https://api.groq.com/openai/v1"` we can use the standard `openai` Python SDK against Groq's hosted models — no Groq-specific SDK required. The `model` field then takes Groq-hosted names like `openai/gpt-oss-120b`.

In [6]:
client = openai.OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)

## Calling LLM without Search MCP Server

We're using `client.responses.create(...)` — the **Responses API** (not Chat Completions). It's the API surface that natively understands hosted MCP tools, which we'll use in the next section.

The query *"Who won the 2026 ICC T20 world cup?"* will fail gracefully here: `gpt-oss-120b`'s training cutoff predates the tournament, so the model can only refuse. That refusal is the motivation for plugging in a search tool.

In [7]:
resp = client.responses.create(
    model="openai/gpt-oss-120b",
    input="Who won the 2026 ICC T20 world cup?",
)

In [8]:
print(resp.output_text)


The 2026 ICC Men’s T20 World Cup hasn’t taken place yet, so there isn’t a champion to name.  

**What we know so far**

| Item | Details |
|------|---------|
| **Scheduled dates** | Late October – early December 2026 (exact dates are still being finalized) |
| **Host nations** | South Africa, Zimbabwe, and Namibia (co‑hosted) |
| **Qualified teams** | The 20‑team format will be used, with the 10 full ICC members automatically qualified and the remaining spots filled via regional qualification tournaments (Asia, Africa, Europe, Americas, and East Pacific) |
| **Format** | Group stage (four groups of five) → Super 12s → knockout (quarter‑finals, semi‑finals, final) |

Because the tournament is still in the future, the winner will be determined only after the final match is played later in 2026. Keep an eye on the ICC’s official website and major sports news outlets as the event approaches—they’ll provide up‑to‑date schedules, team rosters, and, of course, the eventual champion.


## Calling with Tavily Search MCP Server

https://docs.tavily.com/documentation/mcp

**Remote MCP** is the simplest way to give the model a tool: pass the server's URL in the `tools` array and the Responses API handles the entire MCP handshake — discovery, calls, results — server-side, before returning the final answer.

Tavily hosts a managed search MCP server at `https://mcp.tavily.com/mcp/`. Authentication is via a query-string API key. **Never hard-code the key** in a notebook you might share — read it through `getpass` instead, then interpolate it into the URL with an f-string.

Get a free Tavily API key at https://tavily.com (look for *API Keys* in the dashboard).

In [ ]:
tavily_key = getpass("TAVILY_API_KEY: ")

The `server_url` below uses an **f-string** (`f"...{tavily_key}"`) so the secret is interpolated at runtime from the variable we just captured — never written into the notebook source. Each field in the tool dict:

| Field | Purpose |
|-------|---------|
| `type: "mcp"` | Tells the Responses API this is a remote MCP tool. |
| `server_label: "tavily"` | Friendly name shown in the trace; pick anything. |
| `server_url` | Where to reach the MCP server. The Tavily server reads its API key from the query string. |
| `require_approval: "never"` | Auto-approve every tool call. Set to `"always"` if you want to confirm calls manually (useful in human-in-the-loop setups). |

In [ ]:
resp = client.responses.create(
    model="openai/gpt-oss-120b",
    tools=[
        {
            "type": "mcp",
            "server_label": "tavily",
            "server_url": f"https://mcp.tavily.com/mcp/?tavilyApiKey={tavily_key}",
            "require_approval": "never",
        }
    ],
    input="Who won the 2026 ICC T20 world cup?",
)

In [ ]:
print(resp.output_text)

India won the 2026 ICC Men’s T20 World Cup, beating New Zealand in the final at the Narendra Modi Stadium in Ahmedabad. 【2†L4-L9】【2†L19-L24】
